In [169]:
from neo.io import NixIO
import quantities as pq
import numpy as np
import tqdm
import time
import ray
import elephant
from datetime import datetime
import logging
import os
import pickle

filename = "results/network_data27_02_2025.nix"
with NixIO(filename, mode="rw") as io:
    block = io.read_block()
print(block.segments[0].spiketrains[0].times)
print(f"Number of segments: {len(block.segments)}")
print(f"Number of spike_trains: {len(block.segments[0].spiketrains)}")

spiketrains = block.segments[0].spiketrains
metadata = {}
forward_connections = {}  # (source, target) -> delay
forward_sets = {}  # neuron -> set of targets
forward_weights = {} # (source, target) -> weight
side_connections = {} # (source, target) -> delay
side_sets = {} # neuron -> set of targets
side_weights = {} # (source, target) -> weight

n_neurons = 64**2
print(n_neurons)
# Build efficient lookup structures
print("building filtered lookup structures")
for i, values in tqdm.tqdm(enumerate(spiketrains),desc="building filtered lookup structures"):
    metadata[i] = values.annotations
    connections = values.annotations['forward_connections']
    delays = values.annotations['delays']
    connection_weights = values.annotations['weights']
    this_layer_start = (i//n_neurons) * n_neurons
    next_layer_start = this_layer_start + n_neurons
    # Initialize a filtered set for this neuron
    forward_sets[i] = set()
    side_sets[i] = set()
    # Filter connections with weight threshold
    for j, (target, weight) in enumerate(zip(connections, connection_weights)):
        if weight > 0.5:
            if target >= next_layer_start:  # Only keep connections with weights >= 0.5 and in the next layer
                forward_sets[i].add(target)
                forward_connections[(i, target)] = delays[j]
                forward_weights[(i, target)] = weight
            elif target >= this_layer_start: # Only keep connections with weights >= 0.5 and in the same layer
                side_sets[i].add(target)
                side_connections[(i, target)] = delays[j]
                side_weights[(i, target)] = weight
print("done")
print(f"forward len {len(forward_connections)}")
print(f"side len {len(side_connections)}")

triples = []
sparse_triples = []
alpha = 3 * pq.s

# Find triples efficiently with set operations
print("Finding triples with weight-filtered connections...")
for neuron1 in tqdm.tqdm(metadata.keys(), desc="Processing neurons"):
    targets_of_neuron1 = forward_sets[neuron1]
    for neuron2 in targets_of_neuron1:
        t1_2 = forward_connections[(neuron1, neuron2)]
        shared_targets = targets_of_neuron1.intersection(side_sets[neuron2])
        for neuron3 in shared_targets:
            t1_3 = forward_connections[(neuron1, neuron3)]
            t2_3 = side_connections[(neuron2, neuron3)]
            triples.append((neuron1, neuron2, neuron3))
            if abs(t1_2 + t2_3 - t1_3) < alpha:
                sparse_triples.append((neuron1, neuron2, neuron3))

print(f"Found {len(triples)} triples, {len(sparse_triples)} of which are sparse triples")

def add_silent_gaps(spiketrains, segment_duration=250*pq.ms, gap_duration=50*pq.ms):
    modified_spiketrains = []
    total_spikes = sum(len(st) for st in spiketrains)
    print(f"Processing {len(spiketrains)} spike trains with {total_spikes} total spikes")
    print(f"Adding {gap_duration} gaps every {segment_duration}")
    for st in tqdm.tqdm(spiketrains, desc="Processing spike trains"):
        times_ms = st.rescale('ms').magnitude
        segment_duration_ms = segment_duration.rescale('ms').magnitude
        gap_duration_ms = gap_duration.rescale('ms').magnitude
        segments = np.floor(times_ms / segment_duration_ms).astype(int)
        offsets = segments * gap_duration_ms
        new_times_ms = times_ms + offsets
        total_segments = int(np.ceil(st.t_stop.rescale('ms').magnitude / segment_duration_ms))
        new_t_stop_ms = st.t_stop.rescale('ms').magnitude + (total_segments * gap_duration_ms)
        original_units = st.units
        new_st = st.duplicate_with_new_data(
            (new_times_ms * pq.ms).rescale(original_units), 
            t_stop=(new_t_stop_ms * pq.ms).rescale(original_units)
        )
        new_st.annotations.update(st.annotations)
        modified_spiketrains.append(new_st)
    original_duration = max(st.t_stop for st in spiketrains)
    modified_duration = max(st.t_stop for st in modified_spiketrains)
    print(f"Processing complete: Duration extended from {original_duration} to {modified_duration}")
    return modified_spiketrains

modified_spikes = add_silent_gaps(block.segments[0].spiketrains)

test_data = []

def shift_spiketrain(spiketrain, shift):
    new_times = spiketrain.times - shift
    new_times = new_times[new_times >= spiketrain.t_start]
    shifted_st = spiketrain.duplicate_with_new_data(new_times, t_start=spiketrain.t_start, t_stop=spiketrain.t_stop)
    return shifted_st

for i, triple in enumerate(sparse_triples):
    if i % 500 == 0:
        print(f"Processing triple {i}/{len(sparse_triples)}")
    index_1_2 = metadata[triple[0]]['forward_connections'].index(triple[1])
    index_1_3 = metadata[triple[0]]['forward_connections'].index(triple[2])
    index_2_3 = metadata[triple[1]]['forward_connections'].index(triple[2])
    t1_2 = metadata[triple[0]]['delays'][index_1_2] / 1000
    t1_3 = metadata[triple[0]]['delays'][index_1_3] / 1000
    t2_3 = metadata[triple[1]]['delays'][index_2_3] / 1000
    st1 = modified_spikes[triple[0]]
    if len(st1) == 0:
        continue
    st2 = modified_spikes[triple[1]]
    if len(st2) == 0:
        continue
    st3 = modified_spikes[triple[2]]
    if len(st3) == 0:
        continue
    shifted_1 = st1
    shifted_2 = shift_spiketrain(st2, t1_2)
    shifted_3 = shift_spiketrain(st3, (t1_3 + max(t1_2 + t2_3 - t1_3, 0 * pq.s))) # it should have been max(t1_2 + t2_3 - t1_3, t_1_3 * pq.s) Cause they have N past t1_3 to have an impact
    test_data.append((i,{'data': [shifted_1, shifted_2, shifted_3]}))

print(f"Processed {len(test_data)} triples")


In [ ]:
import pickle

# Load the pickled file
with open("azure/processed_results.pkl", "rb") as f:
    analysis_results = pickle.load(f)

print(f"Loaded analysis results with {len(analysis_results)} items")

# # Parse the results into the format needed for the next cell
# local_analysis_data = {}
# for data_index, outcome in analysis_results:
#     triple_signature = sparse_triples[data_index]
#     local_analysis_data[triple_signature] = outcome

# # Extract timings for later use
# timings = {sig: d.timings for sig, d in local_analysis_data.items()}
# print(f"Extracted timing information for {len(timings)} triples")

In [280]:
test_data = []
for i, triple in enumerate(sparse_triples):
    if i % 500 == 0:
        print(f"Processing triple {i}/{len(sparse_triples)}")
    index_1_2 = metadata[triple[0]]['forward_connections'].index(triple[1])
    index_1_3 = metadata[triple[0]]['forward_connections'].index(triple[2])
    index_2_3 = metadata[triple[1]]['forward_connections'].index(triple[2])
    t1_2 = metadata[triple[0]]['delays'][index_1_2] / 1000
    t1_3 = metadata[triple[0]]['delays'][index_1_3] / 1000
    t2_3 = metadata[triple[1]]['delays'][index_2_3] / 1000
    st1 = modified_spikes[triple[0]]
    if len(st1) == 0:
        continue
    st2 = modified_spikes[triple[1]]
    if len(st2) == 0:
        continue
    st3 = modified_spikes[triple[2]]
    if len(st3) == 0:
        continue
    shifted_1 = st1
    shifted_2 = shift_spiketrain(st2, t1_2)
    shifted_3 = shift_spiketrain(st3, (t1_3 + max(t1_2 + t2_3 - t1_3, 0 * pq.s))) # it should have been max(t1_2 + t2_3 - t1_3, t_1_3 * pq.s) Cause they have N past t1_3 to have an impact
    test_data.append((i,{'data': [shifted_1, shifted_2, shifted_3]}))

print(f"Processed {len(test_data)} triples")


In [ ]:
#mid = len(test_data) // 2
#old_test_data, test_data = test_data[:mid], test_data[mid:]


In [ ]:
# suppose I have the pickle file now
local_analysis_data = {}
spike_look_up = {}
for data_index, outcome in analysis_results:
    print(data_index)
    triples_index, spikes = test_data[data_index]
    print(triples_index)
    triple_signature = sparse_triples[triples_index]
    local_analysis_data[triple_signature] = outcome
    spike_look_up[triple_signature] = spikes['data']

print



In [285]:
# Get the first key-value pair from local_analysis_data
first_key, first_value = next(iter(local_analysis_data.items()))

# Print the key (triple) and the structure of the value
print("Key (triple):", first_key)
print(first_value)

In [249]:
significant_patterns = {}
tried = 1
tried2 = 1
for key, item in local_analysis_data.items():
    if item['patterns']:
        # Initialize windows_ids as a Quantity with ms
        windows_ids = pq.Quantity([], units='ms')

        for spectrum in item['patterns']:
            if spectrum['pvalue'] < 0.05:
                if spectrum['neurons'][0] == 0:
                    windows_ids = np.concatenate([windows_ids, spectrum['times'].rescale('ms')])
        if len(windows_ids)>0:
            if tried:
                print(item)
                tried = 0
            significant_patterns[key] = windows_ids
print(f"Found {len(significant_patterns)} significant patterns")
# Update the min and max times across all significant patterns
for pattern_times in significant_patterns.values():
    if len(pattern_times) > 0:
        pattern_min = np.min(pattern_times)
        pattern_max = np.max(pattern_times)
        if tried2:
            min_time = np.min(pattern_times)
            max_time = np.max(pattern_times)
            tried2 = 0
        
        if pattern_min < min_time:
            min_time = pattern_min
        
        if pattern_max > max_time:
            max_time = pattern_max

print(f"Min time: {min_time}")
print(f"Max time: {max_time}")



In [250]:
import numpy as np
import quantities as pq

bin_len = 250
bin_gap = 50
no_epochs = 10
bin_step = bin_len + bin_gap
no_pngs = len(significant_patterns.keys())
start_time = 57600
def bin_pngs(start_time, data, len_stimuli, no_epochs,no_pngs,no_stimuli):
    binned_epoch_data = np.zeros((no_epochs,no_pngs,no_stimuli))
    epoch_len = len_stimuli * no_stimuli
    for tuple_index, (neuron_tuple, png_sigs) in enumerate(data.items()):
        for png_sig in png_sigs:
            shifted_sig = png_sig - start_time
            stimulus_no = int(((shifted_sig % epoch_len)/len_stimuli).magnitude)
            epoch_location = int((shifted_sig/epoch_len).magnitude)
            binned_epoch_data[epoch_location,tuple_index,stimulus_no] +=1
            if tuple_index % 500 == 0:
                print(f"Processing neuron {tuple_index}/{len(data)}")
                print(f"stimulus_no {stimulus_no}")
                print(f"epoch_location {epoch_location}")
    binarised_epoch_data = (binned_epoch_data != 0).astype(int)
    print(binarised_epoch_data.shape)
    f_data = binarised_epoch_data.sum(axis=0)
    return f_data, binarised_epoch_data

f_data_set, binarised_epoch_data_f = bin_pngs(start_time,significant_patterns,bin_step,no_epochs,no_pngs,8)


In [251]:
print(binarised_epoch_data_f.shape)
print(f_data_set.sum(axis=1))

In [ ]:
# import numpy as np
# import quantities as pq

# # Define binning parameters with proper units
# bin_len = 250 * pq.ms
# bin_gap = 50 * pq.ms
# no_epochs = 10
# bin_step = bin_len + bin_gap  # Correct bin step calculation
# start_time = 57600 * pq.ms  # Ensure `start_time` has units

# # Ensure `significant_patterns` exists before calling len()
# if 'significant_patterns' in globals():
#     no_pngs = len(significant_patterns.keys())
# else:
#     raise ValueError("significant_patterns is not defined!")

# def bin_pngs(start_time, data, len_stimuli, no_epochs, no_pngs, no_stimuli):
#     """Bins pattern occurrences into epochs and stimuli categories.
    
#     Parameters:
#         start_time (pq.Quantity): The start time of the epoch.
#         data (dict): Dictionary mapping neuron tuples to spike times.
#         len_stimuli (pq.Quantity): Length of a single stimulus period.
#         no_epochs (int): Number of epochs.
#         no_pngs (int): Number of distinct patterns.
#         no_stimuli (int): Number of stimuli.

#     Returns:
#         np.ndarray: Binned frequency data of patterns per stimulus.
#     """

#     # Ensure `len_stimuli` and `epoch_len` have consistent units
#     if not isinstance(len_stimuli, pq.Quantity):
#         raise TypeError("len_stimuli must be a quantities.Quantity object!")

#     len_stimuli = len_stimuli.rescale('ms')  # Ensure units are ms
#     epoch_len = (len_stimuli * no_stimuli).rescale('ms')  # Ensure correct unit handling

#     # Initialize binned data storage
#     binned_epoch_data = np.zeros((no_epochs, no_pngs, no_stimuli), dtype=int)

#     # Process each pattern
#     for tuple_index, (neuron_tuple, png_sigs) in enumerate(data.items()):
        
#         # Convert spike times into milliseconds, ensuring all have correct units
#         png_sigs = []
#         for sig in png_sigs:
#             if isinstance(sig, pq.Quantity):
#                 png_sigs.append(sig.rescale('ms').magnitude)  # Convert to ms if already a quantity
#             elif isinstance(sig, (int, float)):  
#                 png_sigs.append((sig * pq.ms).magnitude)  # Convert raw numbers to ms
#             else:
#                 raise TypeError(f"Invalid entry in png_sigs: {sig} of type {type(sig)}")  # Catch unexpected types

#         png_sigs = np.array(png_sigs)  # Convert to NumPy array after verification

#                 # Compute relative spike times
#         shifted_sig = png_sigs - start_time.magnitude

#         # Compute stimulus number and epoch location using vectorized operations
#         stimulus_no = np.floor(np.mod(shifted_sig, epoch_len.magnitude) / len_stimuli.magnitude).astype(int)
#         epoch_location = np.floor(shifted_sig / epoch_len.magnitude).astype(int)

#         # Ensure indices are valid before updating array (avoid out-of-bounds errors)
#         valid_mask = (epoch_location >= 0) & (epoch_location < no_epochs) & (stimulus_no >= 0) & (stimulus_no < no_stimuli)
#         np.add.at(binned_epoch_data, (epoch_location[valid_mask], tuple_index, stimulus_no[valid_mask]), 1)

#     # Convert counts to binary presence/absence if needed
#     binarised_epoch_data = (binned_epoch_data != 0).astype(int)

#     # Sum across epochs to get final pattern occurrence count per stimulus
#     f_data = binarised_epoch_data.sum(axis=0)

#     return f_data

# # Ensure `significant_patterns` exists before calling the function
# if 'significant_patterns' not in globals():
#     raise ValueError("significant_patterns is not defined!")

# # Run function with verified inputs
# f_v_data_set = bin_pngs(start_time, significant_patterns, bin_step, no_epochs, no_pngs, 8)


In [235]:
final_layer_indices = [i for i, (key,item) in enumerate(significant_patterns.items()) if key[2] >= 64*64*3]
print(len(final_layer_indices))

In [252]:
binned_data = f_data_set
print(np.sum(binned_data[:,:]))
print(f"Shape of binned data: {binned_data.shape}")
mask = np.ones((binned_data.shape[0], binned_data.shape[1]))
np.fill_diagonal(mask, 0)
true_positives = binned_data
fp_mask = np.ones((binned_data.shape[0], binned_data.shape[1]))
np.fill_diagonal(fp_mask, 0)

# Now use the modified mask
false_positives = binned_data * fp_mask
true_negatives = (no_epochs - binned_data) * mask
false_negatives = no_epochs - binned_data
#there should be 0s need to be able to handle this
precision = true_positives / (true_positives + false_positives)
recall = true_positives / (true_positives + false_negatives)
f1_scores = 2 * (precision * recall) / (precision + recall)

f1_score = np.max(f1_scores, axis=0)
f1_score = np.nan_to_num(f1_score, nan=0.0)
print(np.max(f1_score))
print(np.min(f1_score))
# histogram of f1 scores:
import matplotlib.pyplot as plt
plt.hist(f1_score, bins=50)
plt.xlabel('F1 Score')
plt.ylabel('Frequency')
plt.title('Histogram of F1 Scores')
plt.show()


In [213]:
# Gonna calculate the f1 score for just the first stimulus
print(f"Shape of binned data: {binned_data.shape}")

true_positives = binned_data[:,0]
print(binned_data[0,:])
print(true_positives[0])
adjacents = (binned_data * np.array([[0,1,1,1,1,1,1,1,]]))
false_positives = adjacents.sum(axis=1)
true_negatives = ((no_epochs - binned_data) * np.array([[0,1,1,1,1,1,1,1,]])).sum(axis=1)
print(false_positives[0])
print(true_negatives[0])
false_negatives = no_epochs - binned_data[:,0]
print(false_negatives[0])
precision = true_positives / (true_positives + false_positives)
print(np.min(precision))
recall = true_positives / (true_positives + false_negatives)
print(np.min(recall))
f1_scores = 2 * (precision * recall) / (precision + recall)
f1_scores = np.nan_to_num(f1_scores, nan=0.0)
print((f1_scores[:10]))
print(np.max(f1_scores))
# histogram of f1 scores:
import matplotlib.pyplot as plt
plt.hist(f1_scores, bins=50)
plt.xlabel('F1 Score')
plt.ylabel('Frequency')
plt.title('Histogram of F1 Scores')
plt.show()


In [216]:
import numpy as np
import matplotlib.pyplot as plt

# Assume binned_data is a (PNG_No x Stimulus) array and no_epochs is defined.
# For example:
# binned_data = f_data_set  # shape (PNG_No, Stimulus)
# no_epochs is the total number of epochs used during binning.

print(f"Shape of binned data: {binned_data.shape}")  # e.g., (7541, 8)

# For each neuron (row) and each stimulus (column):
TP = binned_data  # True Positives for each neuron and stimulus

# Compute the row sum (total responses per neuron across all stimuli)
row_sum = np.sum(binned_data, axis=1, keepdims=True)  # shape: (PNG_No, 1)

# False Positives: for each stimulus, the responses in other stimuli.
FP = row_sum - TP  # shape: (PNG_No, Stimulus)

# False Negatives: assume the maximum possible count per stimulus is no_epochs,
# so the misses are:
FN = no_epochs - TP  # shape: (PNG_No, Stimulus)

# Now compute Precision, Recall, and F1 score for each neuron and stimulus.
precision = np.divide(
    TP,
    (TP + FP),
    out=np.zeros_like(TP, dtype=float),
    where=(TP + FP) != 0
)

recall = np.divide(
    TP,
    (TP + FN),
    out=np.zeros_like(TP, dtype=float),
    where=(TP + FN) != 0
)

f1_scores = np.divide(
    2 * (precision * recall),
    (precision + recall),
    out=np.zeros_like(TP, dtype=float),
    where=(precision + recall) != 0
)

# For each neuron, take the maximum F1 score across stimuli.
f1_score_per_neuron = np.max(f1_scores, axis=1)

# Debug prints:
print("Max F1 per neuron:", np.max(f1_score_per_neuron))
print("Min F1 per neuron:", np.min(f1_score_per_neuron))

# Plot a histogram of F1 scores per neuron.
plt.hist(f1_score_per_neuron, bins=50)
plt.xlabel("F1 Score")
plt.ylabel("Frequency")
plt.title("Histogram of F1 Scores per Neuron")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Toy Data Setup ---

# convex: shape (3, 2, 8) describing group membership for 8 patterns.
# For each of 3 definitions, we have 2 groups (e.g. convex vs. concave) across 8 patterns.
convex = np.array([
    [[1,1,1,1,0,0,0,0],
     [0,0,0,0,1,1,1,1]],
    [[1,1,0,0,1,1,0,0],
     [0,0,1,1,0,0,1,1]],
    [[1,0,1,0,1,0,1,0],
     [0,1,0,1,0,1,0,1]]
])
indices_firsts = [[0,1,2,3],[0,1,4,5],[1,3,5,7]]
indices_seconds= [[4,5,6,7],[2,3,6,7],[0,2,4,6]]
results_c = np.zeros((3,len(binned_data),2))

for i, (indices_first, indices_second) in enumerate(zip(indices_firsts, indices_seconds)):

    convex_data = binned_data[:,indices_first].sum(axis=1)
    concave_data = binned_data[:,indices_second].sum(axis=1)
    c_binned_data = np.concatenate([convex_data.reshape(-1,1), concave_data.reshape(-1,1)], axis=1)
    print(c_binned_data.shape)

    # generating f-scores
    true_positives = c_binned_data
    false_positives = c_binned_data[:,::-1]
    true_negatives = no_epochs * 4 - c_binned_data[:,::-1]
    false_negatives = no_epochs * 4 - c_binned_data

    precision = true_positives / (true_positives + false_positives)
    recall = true_positives / (true_positives + false_negatives)
    f1_scores = 2 * (precision * recall) / (precision + recall)
    f1_scores = np.nan_to_num(f1_scores, nan=0.0)
    print(f1_scores.shape)
    print(np.max(f1_scores))
    print(np.min(f1_scores))
    results_c[i,:,:] = f1_scores

# --- Plotting Results ---
f1_max_scores = np.max(results_c, axis = (0,2))
print(f1_max_scores.shape)
# histogram of f1 scores:

# Sort F1 selectivity scores
sorted_f1 = np.sort(f1_max_scores[final_layer_indices])
# Sort F1 selectivity scores in descending order
sorted_f1 = np.sort(f1_max_scores[final_layer_indices])[::-1]

# Calculate the cumulative count (how many neurons have F1 score >= threshold)
neuron_counts = np.arange(1, len(sorted_f1) + 1)

plt.figure(figsize=(10, 6))
plt.plot(neuron_counts,sorted_f1, 'b-', linewidth=2)
plt.grid(True, alpha=0.3)
plt.xscale('log')  # Use log scale on x-axis
plt.ylabel('F1 Score')
plt.xlabel('Number of Neurons (log scale)')
plt.title('Cumulative Distribution of F1 Scores')
plt.show()
sorted_f1_c_indices = np.argsort(f1_max_scores)[::-1]
# top_f1_c = sorted_f1_c_indices[:10]
# print(binned_data[top_f1_c,:])

In [376]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib import colormaps
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.gridspec as gridspec
import matplotlib.collections as mcoll

# Create a custom colormap for the curve gradient
colors = ["#2980b9", "#3498db", "#2ecc71", "#f1c40f", "#e74c3c"]
n_bins = 100
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors, N=n_bins)

# Setup the figure with GridSpec for main plot and histogram
fig = plt.figure(figsize=(12, 8))
gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1])
ax = plt.subplot(gs[0])
ax_hist = plt.subplot(gs[1])

# Calculate the cumulative count
neuron_counts = np.arange(1, len(sorted_f1) + 1)
total_neurons = len(sorted_f1)

# Plot the main curve with gradient coloring
points = np.array([neuron_counts, sorted_f1]).T.reshape(-1, 1, 2)
segments = np.concatenate([points[:-1], points[1:]], axis=1)
norm = plt.Normalize(0, len(segments))
lc = mcoll.LineCollection(segments, cmap=cmap, norm=norm, linewidth=3.5)
lc.set_array(np.linspace(0, 1, len(segments)))
line = ax.add_collection(lc)
ax.set_xlim(neuron_counts.min(), neuron_counts.max())
ax.set_ylim(max(0, sorted_f1.min()-0.05), min(1.0, sorted_f1.max()+0.05))

# Add colored background regions - just two categories now
ax.axhspan(0.5, 1.0, alpha=0.15, color="#2ecc71", label="High selectivity (F1≥0.5)")
ax.axhspan(0.0, 0.5, alpha=0.08, color="#e74c3c", label="Low selectivity (F1<0.5)")

# Annotate key points
highest_f1 = sorted_f1[0]
median_f1 = np.median(sorted_f1)
high_selective = np.sum(sorted_f1 >= 0.5)  # Simplified to just high ≥ 0.5

# Add text annotations with percentages
ax.annotate(f'Highest F1: {highest_f1:.3f}', 
           xy=(1, highest_f1), 
           xytext=(10, highest_f1+0.05),
           arrowprops=dict(arrowstyle='->', lw=1.5, color='#34495e'),
           fontsize=10, color='#34495e')

ax.annotate(f'Median F1: {median_f1:.3f}', 
           xy=(total_neurons/2, median_f1), 
           xytext=(total_neurons/2 + 10, median_f1+0.05),
           arrowprops=dict(arrowstyle='->', lw=1.5, color='#34495e'),
           fontsize=10, color='#34495e')

# Add a reference line at F1 = 0.5
ax.axhline(y=0.5, color='#7f8c8d', linestyle='--', linewidth=1.5, 
          label="Selectivity threshold (F1=0.5)")

# Calculate percentages
high_pct = high_selective/total_neurons*100
low_pct = (total_neurons-high_selective)/total_neurons*100

# Format the axis
ax.grid(True, linestyle='--', alpha=0.7, zorder=0)
ax.set_xscale('log')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='both', which='major', labelsize=10)

# Add better x-axis tick labels
def log_tick_formatter(val, pos=None):
    return f"{int(val):,}" if val >= 1 else "0"
    
ax.xaxis.set_major_formatter(mticker.FuncFormatter(log_tick_formatter))

# Labels and title
ax.set_ylabel('F1 Score', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of PNGs (log scale)', fontsize=14, fontweight='bold')
ax.set_title('Cumulative Distribution of F1 Selectivity Scores', fontsize=16, fontweight='bold', pad=15)

# Add legend with percentages - simpler now
legend_labels = [
    f"High selectivity (F1≥0.5): {high_pct:.1f}%",
    f"Low selectivity (F1<0.5): {low_pct:.1f}%",
    "Selectivity threshold (F1=0.5)"
]
ax.legend(legend_labels, loc='upper right', fontsize=10)

# Add histogram of all F1 scores in lower panel
ax_hist.hist(sorted_f1, bins=30, color='#3498db', alpha=0.7, edgecolor='#2980b9')
ax_hist.set_xlabel('F1 Score', fontsize=12)
ax_hist.set_ylabel('Count', fontsize=12)
ax_hist.spines['top'].set_visible(False)
ax_hist.spines['right'].set_visible(False)
ax_hist.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('f1_score_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# Print out statistics - simplified
print(f"Total PNG's: {total_neurons}")
print(f"Neurons with high selectivity (F1≥0.5): {high_selective} ({high_pct:.1f}%)")
print(f"Neurons with low selectivity (F1<0.5): {total_neurons-high_selective} ({low_pct:.1f}%)")

In [254]:
sums = []
m_sums = []
count = 0
for key, value in significant_patterns.items():
    if count % 400 == 0:
        print(f"Processing triple {count}/{len(significant_patterns)}")
    count+=1
    neuron1, neuron2, neurons3 = key
    t1_2 = forward_connections[(neuron1, neuron2)]
    t1_3 = forward_connections[(neuron1, neurons3)]
    t2_3 = side_connections[(neuron2, neurons3)]
    sums.append(t1_2 + t2_3 - t1_3)
print("got sums")
# now do it for all potential patterns:
count = 0
for i in sparse_triples:
    if count % 400 == 0:
        print(f"Processing triple {count}/{len(sparse_triples)}")
    count+=1
    neuron1, neuron2, neurons3 = i
    t1_2 = forward_connections[(neuron1, neuron2)]
    t1_3 = forward_connections[(neuron1, neurons3)]
    t2_3 = side_connections[(neuron2, neurons3)]
    m_sums.append(t1_2 + t2_3 - t1_3)
print("got m_sums")
# Plot the histogram of sums and m_sums one in blue one in red both semi transparent
plt.hist(sums, bins=50, alpha=0.5, color='blue', label='Significant Patterns')
plt.hist(m_sums, bins=50, alpha=0.5, color='red', label='All Patterns')
plt.xlabel('Sum of Delays')
plt.ylabel('Frequency')
plt.title('Histogram of Delay Sums')
plt.legend()
plt.show()



In [255]:
# Plot the histogram of sums and m_sums one in blue one in red both semi transparent
sums = np.abs(sums)
m_sums = np.abs(m_sums)
plt.hist(sums, bins=50, alpha=0.5, color='blue', label='Significant Patterns')
plt.hist(m_sums, bins=50, alpha=0.5, color='red', label='All Patterns')
plt.xlabel('Sum of Delays')
plt.ylabel('Frequency')
plt.title('Histogram of Delay Sums')
plt.legend()
plt.show()


In [315]:
import viziphant
sorted_f1_c_indices = np.argsort(f1_max_scores)[::-1]
index = sorted_f1_c_indices[0]
# suppose I have the pickle file now
for data_index, outcome in analysis_results:
    if data_index == index:
        triples_index, spikes = test_data[data_index]
        #print(local_analysis_data[key]['patterns'])
        the_spike_trains = spikes['data']
        results_on_spikes = local_analysis_data[key]['patterns']




In [365]:
patterns = results_on_spikes[5]
print(results_on_spikes[5])
start_time = 58 * pq.s
end_time = 61 * pq.s
subset_spiketrains = [st.time_slice(start_time, end_time) for st in the_spike_trains]
print(patterns['times'])
filtered_times = [float((t).rescale('ms').magnitude) for t in patterns['times'] if start_time <= t <= end_time]
filtered_times = pq.Quantity(filtered_times, units='ms')
patterns['lags'][0] += 500 *pq.ms
patterns['lags'][1] += 700 *pq.ms
print(patterns['lags'])

filtered_pattern = {
            'lags': patterns['lags'],
            'neurons': patterns['neurons'],
            'times': filtered_times
        }


import matplotlib.pyplot as plt
import viziphant.patterns
print(subset_spiketrains)
print(filtered_pattern)
# Plot the patterns
axes = viziphant.patterns.plot_patterns(subset_spiketrains, filtered_pattern)
plt.title('Significant Spike Train Patterns')
plt.xlabel('Time (s)')
plt.ylabel('Neuron Index')
plt.savefig('test.png')

In [370]:
import matplotlib.pyplot as plt
import quantities as pq
import numpy as np

# Use a modern style
plt.style.use('seaborn-v0_8-whitegrid')

# Create a single figure with one axis
fig, ax = plt.subplots(figsize=(12, 6))

# Get indices of neurons to plot
neuron_indices = [0, 1, 2]
colors = ['#3498db', '#2ecc71', '#9b59b6']  # Modern colors for each neuron

# Offset for each neuron (vertical position)
offsets = [1, 2, 3]

# Plot all spikes for the three neurons
for i, neuron_idx in enumerate(neuron_indices):
    # Get spike times for this neuron
    spike_times = subset_spiketrains[neuron_idx].rescale('ms').magnitude
    
    # Plot all spikes as small dots
    ax.scatter(spike_times, np.ones_like(spike_times) * offsets[i], 
              s=15, color=colors[i], alpha=0.7, edgecolor='none')

# Now highlight pattern spikes with larger red dots
if len(filtered_pattern['times']) > 0:
    pattern_times = filtered_pattern['times'].rescale('ms').magnitude
    
    # Get which neurons are involved in patterns
    pattern_neurons = filtered_pattern['neurons']
    
    # Highlight spikes involved in patterns for each neuron
    for i, neuron_idx in enumerate(neuron_indices):
        if neuron_idx in pattern_neurons:
            # Determine highlight times based on neuron
            if neuron_idx == 0:
                highlight_times = pattern_times
            elif neuron_idx == 1:
                lag = filtered_pattern['lags'][0].rescale('ms').magnitude
                highlight_times = pattern_times + lag
            elif neuron_idx == 2:
                lag = filtered_pattern['lags'][1].rescale('ms').magnitude
                highlight_times = pattern_times + lag
            
            # Plot pattern spikes as larger red dots with black edge
            ax.scatter(highlight_times, np.ones_like(highlight_times) * offsets[i], 
                      s=80, color='red', edgecolor='black', linewidth=0.5, zorder=10)

# Add neuron labels on the y-axis
ax.set_yticks(offsets)
ax.set_yticklabels([f'Neuron {idx}' for idx in neuron_indices])

# Set axis limits with some padding
ax.set_ylim(0.5, max(offsets) + 0.5)

# Add subtle horizontal lines to separate neurons
for offset in offsets:
    ax.axhline(y=offset - 0.35, color='gray', linestyle='-', alpha=0.2, zorder=0)

# Set labels and title with nicer fonts
ax.set_xlabel('Time (ms)', fontsize=12)
plt.title('Spike Patterns Across Neurons', fontsize=14, fontweight='bold', pad=15)

# Add a legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=colors[i], 
           label=f'Neuron {idx}', markersize=8)
    for i, idx in enumerate(neuron_indices)
]
legend_elements.append(
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red', 
           markeredgecolor='black', label='Pattern spike', markersize=10)
)
ax.legend(handles=legend_elements, loc='upper right', framealpha=0.9)

# Remove top and right spines for cleaner look
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add a very subtle background color
ax.set_facecolor('#f9f9f9')

plt.tight_layout()
plt.savefig('spike_patterns_aesthetic.png', dpi=300)
plt.show()